# NB_LIBRO_06 · Figuras y valores del capítulo 5

**Libro:** *AgroVisión: del píxel a la decisión* · capítulo 5, «Construir un dataset confiable»

**Qué hace.** Muestra los pares de imágenes duplicadas y casi duplicadas del conjunto de palma, y audita las anotaciones y la partición publicada del conjunto PIO de pollos.

**Usa (solo lectura):** `PALMA_AFRICANA_YOLO` (`01_DATOS_ORIGINALES`, `02_DATOS_PROCESADOS`, `03_SPLITS`, `06_RESULTADOS`) y `AgroUTMACH_PIX/01_DATASET_PIO/PIO_TRAIN_VAL/data`. No modifica nada de los proyectos.

**Produce:** seis figuras en `AGROVISION_LIBRO/02_FIGURAS/cap05`, el archivo `AGROVISION_LIBRO/03_VALORES/valores_cap05.json` y dos tablas CSV. Al final descarga todo en un zip.

**Cómo ejecutarlo:** *Entorno de ejecución → Ejecutar todo*. No necesita GPU. Tarda entre 8 y 15 minutos, casi todo en leer las 1 487 imágenes de PIO desde Drive. Suba al chat el zip que se descarga al final (no hace falta el notebook empaquetador).

In [ ]:
import os, re, json, math, zipfile, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))
PALMA = BASE / 'PALMA_AFRICANA_YOLO'
ORIG = PALMA / '01_DATOS_ORIGINALES' / 'zenodo_11114885_extracted'
PIO = BASE / 'AgroUTMACH_PIX' / '01_DATASET_PIO' / 'PIO_TRAIN_VAL' / 'data'
DIR_FIG = BASE / 'AGROVISION_LIBRO' / '02_FIGURAS' / 'cap05'
DIR_VAL = BASE / 'AGROVISION_LIBRO' / '03_VALORES'
for d in (DIR_FIG, DIR_VAL):
    d.mkdir(parents=True, exist_ok=True)

NECESARIOS = [PALMA / '06_RESULTADOS' / 'inventario_imagenes.csv',
              PALMA / '03_SPLITS' / 'palma_split_leakage_safe_seed2026.csv',
              PALMA / '03_SPLITS' / 'palma_processed_manifest_seed2026.csv',
              ORIG, PIO / 'images' / 'train', PIO / 'images' / 'val', PIO / 'labels' / 'train', PIO / 'labels' / 'val']
faltan = [str(p) for p in NECESARIOS if not p.exists()]
assert not faltan, 'No se encuentran estas rutas:\n' + '\n'.join(faltan)
print('Rutas verificadas. Las figuras se guardarán en', DIR_FIG)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from PIL import Image, ImageOps
from skimage.color import rgb2lab, lab2rgb, rgb2hsv

ANCHO_CM = 16.0                      # ancho de caja de texto del libro
ANCHO_IN = ANCHO_CM / 2.54
DPI = 300

# Tinta y colores de apoyo. Los datos categóricos no se distinguen solo por color:
# las cinco clases se muestran en paneles separados (Figura 2.7).
TINTA = '#0b0b0b'
TINTA_2 = '#52514e'
GRIS_FONDO = '#c9c8c2'
ACENTO = '#2a78d6'
REJILLA = '#e4e3de'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA,
    'text.color': TINTA,
    'xtick.color': TINTA_2,
    'ytick.color': TINTA_2,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': DPI,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

CLASES = ['unripe', 'ripe', 'overripe', 'damaged', 'empty']       # orden de presentación
NOMBRE_ES = {'unripe': 'Inmaduro', 'ripe': 'Maduro', 'overripe': 'Sobremaduro',
             'damaged': 'Dañado', 'empty': 'Vacío'}
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

VALORES = {}          # cifras que se copian al manuscrito
FIGURAS = []          # rutas de las figuras generadas


def guardar(fig, ruta):
    # Guarda la figura a 300 dpi y la registra.
    fig.savefig(ruta, dpi=DPI, bbox_inches='tight', pad_inches=0.04)
    FIGURAS.append(str(ruta))
    print('Guardada:', ruta.name)


def leer_rgb(ruta, lado_max=None):
    # Lee una imagen como arreglo RGB uint8, respetando la orientación EXIF.
    # Con lado_max, la reduce para que su lado mayor no supere ese valor.
    with Image.open(ruta) as im:
        if lado_max is not None and im.format == 'JPEG':
            im.draft('RGB', (lado_max * 2, lado_max * 2))     # decodificación rápida
        im = ImageOps.exif_transpose(im).convert('RGB')
        if lado_max is not None and max(im.size) > lado_max:
            im.thumbnail((lado_max, lado_max), Image.LANCZOS)
        return np.asarray(im)


def redimensionar(img, lado_mayor):
    # Redimensiona conservando la proporción hasta que el lado mayor mida lado_mayor.
    h, w = img.shape[:2]
    escala = lado_mayor / max(h, w)
    nuevo = (max(1, round(w * escala)), max(1, round(h * escala)))
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, nuevo, interpolation=interp)


def recorte_cuadrado(img, lado=512):
    # Recorte cuadrado centrado, redimensionado a lado × lado píxeles.
    h, w = img.shape[:2]
    m = min(h, w)
    y0, x0 = (h - m) // 2, (w - m) // 2
    return cv2.resize(img[y0:y0 + m, x0:x0 + m], (lado, lado), interpolation=cv2.INTER_AREA)


def coma(x, dec=1):
    # Número con coma decimal, como se escribe en español.
    return f'{x:.{dec}f}'.replace('.', ',')


def eje_con_coma(eje, dec=2):
    eje.set_major_formatter(FuncFormatter(lambda v, _: coma(v, dec)))


def sin_ejes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


print('Bibliotecas listas. matplotlib', matplotlib.__version__, '| OpenCV', cv2.__version__)

## 1. Palma: copias exactas y casi duplicados

Las tablas del proyecto ya contienen la huella SHA-256 de cada archivo y el dHash de 64 bits de cada imagen. Aquí solo se leen y se muestran los pares.

In [ ]:
inv = pd.read_csv(PALMA / '06_RESULTADOS' / 'inventario_imagenes.csv')
sp = pd.read_csv(PALMA / '03_SPLITS' / 'palma_split_leakage_safe_seed2026.csv')
man = pd.read_csv(PALMA / '03_SPLITS' / 'palma_processed_manifest_seed2026.csv')
PROC = dict(zip(man['sha256_source'], man['processed_relative_path']))
CARPETA = {'train_original': 'entrenamiento', 'test_original': 'prueba'}
PART_ES = {'train': 'entrenamiento', 'val': 'validación', 'test': 'prueba'}


def img_orig(rel, lado=360):
    return recorte_cuadrado(leer_rgb(ORIG / rel, 720), lado)


def img_proc(sha, lado=360):
    return recorte_cuadrado(leer_rgb(PALMA / PROC[sha], 720), lado)


def pie(fila):
    # Texto bajo cada imagen: nombre del archivo, carpeta publicada y tamaño original.
    return f"{Path(fila['filename']).stem}\n{CARPETA[fila['split_inferred']]}\n{int(fila['width'])} × {int(fila['height'])}"


def figura_pares(pares, n_filas, n_cols, nombre, alto_fila=1.62):
    # pares: lista de dicts con las claves a, b (imágenes), ta, tb (pies) y cab (encabezado).
    fig = plt.figure(figsize=(ANCHO_IN, alto_fila * n_filas))
    gs = fig.add_gridspec(n_filas, n_cols, wspace=0.10, hspace=0.72, left=0.01, right=0.99, top=0.93, bottom=0.13)
    for k, par in enumerate(pares[:n_filas * n_cols]):
        sub = gs[k // n_cols, k % n_cols].subgridspec(1, 2, wspace=0.03)
        for m, (clave_img, clave_txt) in enumerate((('a', 'ta'), ('b', 'tb'))):
            ax = fig.add_subplot(sub[0, m])
            ax.imshow(par[clave_img]); sin_ejes(ax)
            ax.set_xlabel(par[clave_txt], fontsize=5.6, color=TINTA_2, labelpad=2)
            if m == 0:
                ax.set_title(par['cab'], loc='left', fontsize=7, fontweight='bold', pad=3)
    ruta = DIR_FIG / nombre
    guardar(fig, ruta); plt.show()


def hamming(A, B):
    # Matriz de distancias de Hamming entre dos vectores de huellas de 64 bits.
    x = (A[:, None] ^ B[None, :])
    return np.unpackbits(x.view(np.uint8).reshape(len(A), len(B), 8), axis=2).sum(2).astype(np.int16)


# --- copias exactas ---
grupos = [g.sort_values('relative_path') for _, g in inv.groupby('sha256') if len(g) > 1]
cruzan = [g for g in grupos if g['split_inferred'].nunique() > 1]
VALORES['copias_exactas'] = {
    'grupos': len(grupos), 'archivos_en_grupos': int(sum(len(g) for g in grupos)),
    'sobrantes': int(sum(len(g) - 1 for g in grupos)), 'grupos_que_cruzan_carpetas': len(cruzan),
    'lista': [[Path(f).stem for f in g['filename']] for g in grupos]}
print(json.dumps(VALORES['copias_exactas'], ensure_ascii=False)[:600])

# --- casi duplicados dentro de la misma clase ---
H = np.array([int(h, 16) for h in sp['dhash_hex']], dtype=np.uint64)
D = hamming(H, H)
clase = sp['class_inferred'].to_numpy(dtype=object)
misma = clase[:, None] == clase[None, :]
ii, jj = np.triu_indices(len(sp), 1)
pares_df = pd.DataFrame({'i': ii, 'j': jj, 'd': D[ii, jj], 'misma': misma[ii, jj]})
pares_df['cruza'] = sp['split_inferred'].to_numpy(dtype=object)[ii] != sp['split_inferred'].to_numpy(dtype=object)[jj]
cerca = pares_df[(pares_df['d'] <= 4) & pares_df['misma']]
VALORES['casi_duplicados'] = {
    'pares_le4': int(len(cerca)), 'pares_que_cruzan_carpetas': int(cerca['cruza'].sum()),
    'por_distancia': {int(k): int(v) for k, v in cerca['d'].value_counts().sort_index().items()},
    'pares_misma_clase_entre_5_y_10': int(((pares_df['d'] >= 5) & (pares_df['d'] <= 10) & pares_df['misma']).sum())}
print(VALORES['casi_duplicados'])

In [ ]:
# Figura 5.2 · Arriba, copias exactas que cruzan las carpetas publicadas; abajo, casi duplicados.
pares = []
for g in cruzan[:3]:
    a = g[g['split_inferred'] == 'train_original'].iloc[0]; b = g[g['split_inferred'] == 'test_original'].iloc[0]
    pares.append({'a': img_orig(a['relative_path']), 'b': img_orig(b['relative_path']), 'ta': pie(a), 'tb': pie(b),
                  'cab': 'Copia exacta (SHA-256 idéntico)'})
elegidos = []
for d_obj in (0, 2, 4):
    cand = cerca[(cerca['d'] == d_obj) & ~cerca['i'].isin([e[0] for e in elegidos])].sort_values('cruza', ascending=False)
    if len(cand):
        f = cand.iloc[0]; elegidos.append((int(f['i']), int(f['j']), int(f['d'])))
for i, j, d in elegidos:
    a, b = sp.iloc[i], sp.iloc[j]
    pares.append({'a': img_proc(a['sha256']), 'b': img_proc(b['sha256']), 'ta': pie(a), 'tb': pie(b),
                  'cab': f'Casi duplicado · distancia {d}'})
VALORES['fig_5_2'] = [{'cab': p['cab'], 'a': p['ta'].replace('\n', ' | '), 'b': p['tb'].replace('\n', ' | ')} for p in pares]
figura_pares(pares, 2, 3, 'Fig_5_2_duplicados.png')

In [ ]:
# Figura 5.4 · ¿Dónde poner el umbral? Pares de la misma clase a distancias crecientes.
pares, usados = [], set()
for d_obj in (3, 5, 7, 9, 11, 14):
    cand = pares_df[pares_df['misma'] & (pares_df['d'] >= d_obj) & ~pares_df['i'].isin(usados) & ~pares_df['j'].isin(usados)]
    cand = cand.sort_values(['d', 'i', 'j'])
    if not len(cand):
        continue
    f = cand.iloc[0]; i, j = int(f['i']), int(f['j']); usados |= {i, j}
    a, b = sp.iloc[i], sp.iloc[j]
    pares.append({'a': img_proc(a['sha256']), 'b': img_proc(b['sha256']), 'ta': pie(a), 'tb': pie(b),
                  'cab': f"Distancia {int(f['d'])}"})
VALORES['fig_5_4'] = [{'cab': p['cab'], 'a': p['ta'].replace('\n', ' | '), 'b': p['tb'].replace('\n', ' | ')} for p in pares]
figura_pares(pares, 2, 3, 'Fig_5_4_umbral_pares.png')

In [ ]:
# Figura 5.5 · Pares casi idénticos según el dHash pero con etiquetas de clase distintas.
entre = pares_df[(pares_df['d'] <= 4) & ~pares_df['misma']].sort_values('d')
pares, info = [], []
for _, f in entre.head(3).iterrows():
    a, b = sp.iloc[int(f['i'])], sp.iloc[int(f['j'])]
    ga = cv2.resize(cv2.cvtColor(leer_rgb(PALMA / PROC[a['sha256']], 256), cv2.COLOR_RGB2GRAY), (64, 64)).astype(float)
    gb = cv2.resize(cv2.cvtColor(leer_rgb(PALMA / PROC[b['sha256']], 256), cv2.COLOR_RGB2GRAY), (64, 64)).astype(float)
    r = float(np.corrcoef(ga.ravel(), gb.ravel())[0, 1])
    pares.append({'a': recorte_cuadrado(leer_rgb(PALMA / PROC[a['sha256']], 720), 360),
                  'b': recorte_cuadrado(leer_rgb(PALMA / PROC[b['sha256']], 720), 360),
                  'ta': f"Etiqueta: {NOMBRE_ES[a['class_inferred']].lower()}\n{pie(a)}", 'tb': f"Etiqueta: {NOMBRE_ES[b['class_inferred']].lower()}\n{pie(b)}",
                  'cab': f"Distancia {int(f['d'])} · clases distintas"})
    info.append({'a': a['filename'], 'b': b['filename'], 'clase_a': a['class_inferred'], 'clase_b': b['class_inferred'],
                 'd': int(f['d']), 'correlacion_64px': round(r, 3), 'particion_a': a['split'], 'particion_b': b['split']})
VALORES['pares_entre_clases'] = {'n_le4': int(len(entre)), 'detalle': info}
print(json.dumps(info, ensure_ascii=False, indent=1))
if pares:
    figura_pares(pares, 1, max(2, len(pares)), 'Fig_5_5_pares_entre_clases.png', alto_fila=2.1 if len(pares) < 3 else 1.62)

## 2. Pollos: auditoría de las anotaciones del conjunto PIO

Se leen todos los archivos de etiquetas y se comprueba, línea por línea, que cumplan el formato YOLO: cinco números, la clase y cuatro coordenadas normalizadas entre 0 y 1.

In [ ]:
PATRON = re.compile(r'^([CP])-W(\d+)-(V?)(\d+)$', re.I)
GALPON = {'C': 'Comercial', 'P': 'Prototipo'}
ANCHO_PX, ALTO_PX = 1920, 1080
reg, cajas, problemas, huerfanas = [], [], [], []
for part in ('train', 'val'):
    imgs = {p.stem: p for p in sorted((PIO / 'images' / part).iterdir()) if p.suffix.lower() in EXT_IMG}
    txts = {p.stem: p for p in sorted((PIO / 'labels' / part).glob('*.txt'))}
    for stem in sorted(set(txts) - set(imgs)):
        huerfanas.append({'particion': part, 'archivo': txts[stem].name,
                          'contenido': txts[stem].read_text(encoding='utf-8', errors='ignore')[:200]})
    for stem, ruta in imgs.items():
        h = PATRON.match(stem)
        fila = {'particion': part, 'archivo': ruta.name, 'ruta': str(ruta), 'galpon': h.group(1).upper() if h else '?',
                'semana': int(h.group(2)) if h else 0, 'con_etiqueta': stem in txts, 'cajas': 0, 'repetidas': 0,
                'fuera_de_rango': 0, 'sobresalen': 0, 'mal_formadas': 0}
        if stem in txts:
            buenas = []
            for n_linea, linea in enumerate(txts[stem].read_text(encoding='utf-8', errors='ignore').splitlines(), 1):
                partes = linea.split()
                if not partes:
                    continue
                try:
                    v = [float(x) for x in partes]
                    assert len(v) == 5
                    buenas.append(v)
                except Exception:
                    fila['mal_formadas'] += 1
                    problemas.append({'archivo': txts[stem].name, 'linea': n_linea, 'texto': linea[:80]})
            a = np.array(buenas, dtype=float).reshape(-1, 5)
            fila['cajas'] = len(a)
            if len(a):
                fila['repetidas'] = int(len(a) - len(np.unique(a, axis=0)))
                fila['fuera_de_rango'] = int(((a[:, 1:] < 0) | (a[:, 1:] > 1)).any(axis=1).sum())
                x1, x2 = a[:, 1] - a[:, 3] / 2, a[:, 1] + a[:, 3] / 2
                y1, y2 = a[:, 2] - a[:, 4] / 2, a[:, 2] + a[:, 4] / 2
                fila['sobresalen'] = int(((x1 < -1e-3) | (y1 < -1e-3) | (x2 > 1 + 1e-3) | (y2 > 1 + 1e-3)).sum())
                cajas.append(np.column_stack([np.full(len(a), len(reg)), a]))
        reg.append(fila)
pio = pd.DataFrame(reg)
cj = np.vstack(cajas)                       # columnas: índice de imagen, clase, xc, yc, w, h
bx = pd.DataFrame({'img': cj[:, 0].astype(int), 'clase': cj[:, 1].astype(int), 'xc': cj[:, 2], 'yc': cj[:, 3],
                   'w_px': cj[:, 4] * ANCHO_PX, 'h_px': cj[:, 5] * ALTO_PX})
bx['lado'] = np.sqrt(bx['w_px'] * bx['h_px'])
bx['semana'] = pio['semana'].to_numpy()[bx['img']]; bx['galpon'] = pio['galpon'].to_numpy()[bx['img']]
bx['particion'] = pio['particion'].to_numpy()[bx['img']]


def resumen(g):
    return {'imagenes': int(len(g)), 'cajas': int(g['cajas'].sum()), 'media': round(float(g['cajas'].mean()), 1),
            'mediana': float(g['cajas'].median()), 'min': int(g['cajas'].min()), 'max': int(g['cajas'].max())}


def cuantiles(s):
    return {k: round(float(s.quantile(q)), 1) for k, q in (('p5', .05), ('p25', .25), ('p50', .5), ('p75', .75), ('p95', .95))}


yaml_txt = (PIO / 'dataset.yaml').read_text(encoding='utf-8', errors='ignore') if (PIO / 'dataset.yaml').exists() else ''
VALORES['pio_auditoria'] = {
    'imagenes': pio.groupby('particion').size().to_dict(), 'imagenes_sin_etiqueta': int((~pio['con_etiqueta']).sum()),
    'etiquetas_sin_imagen': huerfanas, 'lineas_mal_formadas': problemas[:20], 'n_mal_formadas': len(problemas),
    'cajas_validas': int(len(bx)), 'clases': {int(k): int(v) for k, v in bx['clase'].value_counts().items()},
    'cajas_repetidas': int(pio['repetidas'].sum()), 'imagenes_con_repetidas': int((pio['repetidas'] > 0).sum()),
    'cajas_fuera_de_rango': int(pio['fuera_de_rango'].sum()), 'cajas_que_sobresalen': int(pio['sobresalen'].sum()),
    'imagenes_sin_cajas': int(((pio['cajas'] == 0) & pio['con_etiqueta']).sum()),
    'nombres_no_reconocidos': int((pio['galpon'] == '?').sum()), 'dataset_yaml': yaml_txt[:300],
    'otros_archivos': sorted(p.name for p in PIO.iterdir() if p.is_file()),
    'por_particion': {k: resumen(g) for k, g in pio.groupby('particion')},
    'por_particion_galpon_semana': {f'{p}|{ga}|{s}': resumen(g) for (p, ga, s), g in pio.groupby(['particion', 'galpon', 'semana'])},
    'lado_caja_px': {'todas': cuantiles(bx['lado']), 'por_semana': {int(s): cuantiles(g['lado']) for s, g in bx.groupby('semana')},
                     'por_galpon': {k: cuantiles(g['lado']) for k, g in bx.groupby('galpon')}},
    'ancho_px': cuantiles(bx['w_px']), 'alto_px': cuantiles(bx['h_px']),
    'pequenas_coco_original_pct': round(float((bx['lado'] < 32).mean() * 100), 1),
    'pequenas_coco_a_960_pct': round(float((bx['lado'] * 0.5 < 32).mean() * 100), 1),
    'medianas_coco_a_960_pct': round(float(((bx['lado'] * 0.5 >= 32) & (bx['lado'] * 0.5 < 96)).mean() * 100), 1),
    'cajas_menores_de_8px_a_960': int((bx['lado'] * 0.5 < 8).sum())}
v = VALORES['pio_auditoria']
print({k: v[k] for k in ('imagenes', 'imagenes_sin_etiqueta', 'n_mal_formadas', 'cajas_validas', 'cajas_repetidas',
                          'cajas_fuera_de_rango', 'cajas_que_sobresalen', 'imagenes_sin_cajas', 'otros_archivos')})
print('Etiquetas sin imagen:', huerfanas); print('Líneas mal formadas:', problemas[:5])
print(pio.pivot_table(index=['particion', 'galpon'], columns='semana', values='cajas', aggfunc='count', fill_value=0).to_string())
pio.drop(columns='ruta').to_csv(DIR_VAL / 'pio_auditoria_por_imagen.csv', index=False)

In [ ]:
# Figura 5.8 · El formato YOLO sobre una imagen real: todas las cajas y una de ellas en detalle.
cand = pio[(pio['particion'] == 'val') & (pio['semana'] == pio['semana'].max()) & (pio['cajas'] >= 15)].sort_values(['cajas', 'archivo'])
if not len(cand):
    cand = pio[pio['cajas'] > 0].sort_values(['cajas', 'archivo'])
ej = cand.iloc[0]
img = leer_rgb(ej['ruta']); alto, ancho = img.shape[:2]
b = bx[bx['img'] == ej.name].copy()
b['dist'] = np.hypot(b['xc'] - 0.5, b['yc'] - 0.5) + 2 * np.abs(b['lado'] / b['lado'].median() - 1)
c = b.sort_values('dist').iloc[0]
xc, yc, wn, hn = float(c['xc']), float(c['yc']), float(c['w_px']) / ANCHO_PX, float(c['h_px']) / ALTO_PX
X, Y, Wp, Hp = xc * ancho, yc * alto, wn * ancho, hn * alto

fig = plt.figure(figsize=(ANCHO_IN, 2.75))
gs = fig.add_gridspec(1, 2, width_ratios=[1.78, 1], wspace=0.04, left=0.005, right=0.995, top=0.92, bottom=0.2)
a1 = fig.add_subplot(gs[0, 0]); a1.imshow(img); sin_ejes(a1)
for _, q in b.iterrows():
    w_, h_ = q['w_px'] / ANCHO_PX * ancho, q['h_px'] / ALTO_PX * alto
    a1.add_patch(Rectangle((q['xc'] * ancho - w_ / 2, q['yc'] * alto - h_ / 2), w_, h_, fill=False, ec='white', lw=0.5))
a1.add_patch(Rectangle((X - Wp / 2, Y - Hp / 2), Wp, Hp, fill=False, ec=ACENTO, lw=1.4))
a1.set_title(f"a) {ej['archivo']}: {len(b)} cajas anotadas", loc='left', fontweight='bold')
m = 1.6 * max(Wp, Hp)
x0, y0 = int(np.clip(X - m, 0, ancho - 2 * m)), int(np.clip(Y - m, 0, alto - 2 * m))
a2 = fig.add_subplot(gs[0, 1]); a2.imshow(img[y0:y0 + int(2 * m), x0:x0 + int(2 * m)]); sin_ejes(a2)
u, w_ = X - x0, Y - y0
a2.add_patch(Rectangle((u - Wp / 2, w_ - Hp / 2), Wp, Hp, fill=False, ec=ACENTO, lw=1.6))
a2.plot([u], [w_], marker='+', color='white', ms=9, mew=1.6)
caja_txt = dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.85)
a2.annotate('', xy=(u - Wp / 2, w_ + Hp / 2 + 0.16 * m), xytext=(u + Wp / 2, w_ + Hp / 2 + 0.16 * m),
            arrowprops=dict(arrowstyle='<->', color='white', lw=1.1))
a2.text(u, w_ + Hp / 2 + 0.34 * m, 'ancho', ha='center', va='center', fontsize=7, bbox=caja_txt)
a2.annotate('', xy=(u + Wp / 2 + 0.16 * m, w_ - Hp / 2), xytext=(u + Wp / 2 + 0.16 * m, w_ + Hp / 2),
            arrowprops=dict(arrowstyle='<->', color='white', lw=1.1))
a2.text(u + Wp / 2 + 0.3 * m, w_, 'alto', ha='left', va='center', fontsize=7, bbox=caja_txt)
a2.text(u, w_ - Hp / 2 - 0.2 * m, 'centro (x, y)', ha='center', va='center', fontsize=7, bbox=caja_txt)
a2.set_title('b) Una caja en detalle', loc='left', fontweight='bold')
linea = f'0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}'
fig.text(0.005, 0.105, 'Línea del archivo de etiquetas:', fontsize=7.5, color=TINTA_2)
fig.text(0.27, 0.105, linea, fontsize=7.5, family='DejaVu Sans Mono', color=ACENTO, fontweight='bold')
fig.text(0.005, 0.03, f'En píxeles ({ancho} × {alto}): centro en ({X:.0f}, {Y:.0f}); ancho {Wp:.0f}; alto {Hp:.0f}', fontsize=7.5, color=TINTA_2)
guardar(fig, DIR_FIG / 'Fig_5_8_formato_yolo.png'); plt.show()
VALORES['fig_5_8'] = {'archivo': ej['archivo'], 'cajas': int(len(b)), 'linea': linea, 'imagen_px': [int(ancho), int(alto)],
                      'centro_px': [round(X), round(Y)], 'ancho_px': round(Wp), 'alto_px': round(Hp)}
print(VALORES['fig_5_8'])

In [ ]:
# Figura 5.9 · Composición del conjunto PIO y tamaño de las aves anotadas.
semanas = sorted(s for s in pio['semana'].unique() if s > 0)
fig = plt.figure(figsize=(ANCHO_IN, 4.5))
gs = fig.add_gridspec(2, 2, height_ratios=[0.8, 1], hspace=0.62, wspace=0.28)
a0 = fig.add_subplot(gs[0, :])
filas_t = [(g, p) for g in ('C', 'P') for p in ('train', 'val')]
tabla = np.array([[int(((pio['galpon'] == g) & (pio['particion'] == p) & (pio['semana'] == s)).sum()) for s in semanas] for g, p in filas_t])
a0.imshow(tabla, cmap=LinearSegmentedColormap.from_list('a', ['#ffffff', ACENTO]), vmin=0, vmax=max(1, tabla.max()) * 1.25, aspect='auto')
for r in range(tabla.shape[0]):
    for k in range(tabla.shape[1]):
        a0.text(k, r, str(tabla[r, k]), ha='center', va='center', fontsize=7.5,
                color='white' if tabla[r, k] > 0.62 * tabla.max() else TINTA)
    a0.text(len(semanas) - 0.42, r, f'{tabla[r].sum()}', ha='left', va='center', fontsize=7.5, fontweight='bold')
a0.set_xticks(range(len(semanas))); a0.set_xticklabels([f'Semana {s}' for s in semanas])
a0.set_yticks(range(4)); a0.set_yticklabels([f"{GALPON[g]} · {PART_ES[p]}" for g, p in filas_t])
a0.tick_params(length=0); a0.xaxis.tick_top()
for s_ in a0.spines.values():
    s_.set_visible(False)
a0.set_xticks(np.arange(-.5, len(semanas)), minor=True); a0.set_yticks(np.arange(-.5, 4), minor=True)
a0.grid(which='minor', color='white', lw=1.5); a0.tick_params(which='minor', length=0)
a0.set_title('a) Imágenes por galpón, partición y semana de edad', loc='left', fontweight='bold', pad=16)
a0.text(len(semanas) - 0.42, -0.78, 'Total', ha='left', va='center', fontsize=7, color=TINTA_2)

ejes = [fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]
for ax, (col, datos, rotulo, titulo) in zip(ejes, (('cajas', pio, 'Aves anotadas por imagen', 'b) Aves por imagen'),
                                                   ('lado', bx, 'Lado equivalente de la caja (píxeles)', 'c) Tamaño de las aves'))):
    for desp, g, color in ((-0.17, 'C', ACENTO), (0.17, 'P', TINTA_2)):
        for s in semanas:
            x = datos[(datos['galpon'] == g) & (datos['semana'] == s)][col]
            if not len(x):
                continue
            q = x.quantile([.05, .25, .5, .75, .95]).to_numpy()
            ax.plot([s + desp] * 2, [q[0], q[4]], color=color, lw=0.8)
            ax.add_patch(Rectangle((s + desp - 0.13, q[1]), 0.26, q[3] - q[1], fc=color, ec='none', alpha=0.85))
            ax.plot([s + desp - 0.13, s + desp + 0.13], [q[2], q[2]], color='white', lw=1.1)
        ax.plot([], [], color=color, lw=5, label=f'Galpón {GALPON[g].lower()}')
    ax.set_xticks(semanas); ax.set_xlabel('Semana de edad'); ax.set_ylabel(rotulo); ax.set_ylim(bottom=0)
    ax.yaxis.grid(True, color=REJILLA, lw=0.6); ax.set_axisbelow(True)
    ax.set_title(titulo, loc='left', fontweight='bold')
ejes[0].legend(frameon=False, loc='upper left', handlelength=1)
guardar(fig, DIR_FIG / 'Fig_5_9_pio_composicion.png'); plt.show()

## 3. Pollos: ¿qué tan independiente es la validación publicada?

Cada imagen de validación se compara con todas las de entrenamiento mediante una miniatura en grises de 64 × 36 píxeles. La distancia es la diferencia absoluta media entre miniaturas, en niveles de gris. Como referencia se calcula la misma distancia dentro del conjunto de entrenamiento.

In [ ]:
def miniatura(ruta):
    with Image.open(ruta) as im:
        tam = im.size
        im.draft('L', (256, 144))
        g = np.asarray(im.convert('L').resize((64, 36), Image.BILINEAR), dtype=np.float32)
    return tam, g

t0 = time.time(); minis, tamanos = [], []
for k, ruta in enumerate(pio['ruta'], 1):
    tam, g = miniatura(ruta); minis.append(g.ravel()); tamanos.append(tam)
    if k % 200 == 0 or k == len(pio):
        print(f'Miniaturas: {k}/{len(pio)} · {time.time() - t0:.0f} s', end='\r')
print()
M = np.vstack(minis); pio['ancho'] = [t[0] for t in tamanos]; pio['alto'] = [t[1] for t in tamanos]
i_tr = np.where(pio['particion'] == 'train')[0]; i_va = np.where(pio['particion'] == 'val')[0]


def vecino(A, B, excluir_diagonal=False):
    # Para cada fila de A, la fila de B con menor diferencia absoluta media.
    idx, dist = np.zeros(len(A), int), np.zeros(len(A), np.float32)
    for k in range(0, len(A), 16):
        d = np.abs(A[k:k + 16, None, :] - B[None, :, :]).mean(2)
        if excluir_diagonal:
            d[np.arange(d.shape[0]), np.arange(k, k + d.shape[0])] = np.inf
        idx[k:k + 16] = d.argmin(1); dist[k:k + 16] = d.min(1)
    return idx, dist

j_va, d_va = vecino(M[i_va], M[i_tr])
j_tr, d_tr = vecino(M[i_tr], M[i_tr], excluir_diagonal=True)
vec = pio.iloc[i_va][['archivo', 'galpon', 'semana', 'cajas']].reset_index(drop=True)
vt = pio.iloc[i_tr[j_va]].reset_index(drop=True)
vec['vecino_train'] = vt['archivo']; vec['cajas_vecino'] = vt['cajas']; vec['dist'] = d_va
vec['mismo_galpon_semana'] = (vec['galpon'] == vt['galpon']) & (vec['semana'] == vt['semana'])
vec.to_csv(DIR_VAL / 'pio_vecino_val_train.csv', index=False)
VALORES['pio_val_train'] = {
    'tamanos_imagen': {f'{a}x{b}': int(n) for (a, b), n in pio.groupby(['ancho', 'alto']).size().items()},
    'dist_val_a_train': cuantiles(pd.Series(d_va)) | {'min': round(float(d_va.min()), 2), 'max': round(float(d_va.max()), 2)},
    'dist_train_a_train': cuantiles(pd.Series(d_tr)) | {'min': round(float(d_tr.min()), 2), 'max': round(float(d_tr.max()), 2)},
    'val_con_vecino_mismo_galpon_semana_pct': round(float(vec['mismo_galpon_semana'].mean() * 100), 1),
    'val_con_dist_menor_2': int((d_va < 2).sum()), 'val_con_dist_menor_4': int((d_va < 4).sum()),
    'val_con_dist_menor_8': int((d_va < 8).sum()), 'train_con_dist_menor_4': int((d_tr < 4).sum()),
    'dif_cajas_con_vecino_mediana': float((vec['cajas'] - vec['cajas_vecino']).abs().median()),
    'por_galpon': {g: cuantiles(x['dist']) for g, x in vec.groupby('galpon')}}
print(json.dumps(VALORES['pio_val_train'], ensure_ascii=False, indent=1))

In [ ]:
# Figura 5.7 · Imágenes de validación (arriba) y su imagen más parecida de entrenamiento (abajo).
orden = np.argsort(d_va)
elegidas = [orden[0], orden[len(orden) // 4], orden[len(orden) // 2], orden[-1]]
rotulos = ['La más cercana', 'Percentil 25', 'Mediana', 'La más lejana']
fig = plt.figure(figsize=(ANCHO_IN, 4.4))
ext = fig.add_gridspec(2, 1, height_ratios=[2.25, 1], hspace=0.2, left=0.07, right=0.995, top=0.94, bottom=0.09)
gs = ext[0].subgridspec(2, 4, hspace=0.16, wspace=0.05)
detalle = []
for k, (e, rot) in enumerate(zip(elegidas, rotulos)):
    fv, ft = pio.iloc[i_va[e]], pio.iloc[i_tr[j_va[e]]]
    for r, f in enumerate((fv, ft)):
        ax = fig.add_subplot(gs[r, k]); ax.imshow(leer_rgb(f['ruta'], 480)); sin_ejes(ax)
        ax.set_xlabel(f"{Path(f['archivo']).stem} · {int(f['cajas'])} aves", fontsize=5.8, color=TINTA_2, labelpad=2)
        if r == 0:
            ax.set_title(f'{rot}\ndiferencia {coma(float(d_va[e]), 1)}', fontsize=7, fontweight='bold', loc='left', pad=3)
        if k == 0:
            ax.set_ylabel('Validación' if r == 0 else 'Entrenamiento', fontsize=7)
    detalle.append({'rotulo': rot, 'val': fv['archivo'], 'train': ft['archivo'], 'dist': round(float(d_va[e]), 2),
                    'cajas_val': int(fv['cajas']), 'cajas_train': int(ft['cajas'])})
ax = fig.add_subplot(ext[1])
tope = float(np.percentile(np.r_[d_va, d_tr], 99.5)); bordes = np.linspace(0, tope, 41)
ax.hist(np.clip(d_va, 0, tope), bins=bordes, weights=np.full(len(d_va), 100 / len(d_va)), color=ACENTO, alpha=0.9,
        label=f'Validación → entrenamiento ({len(d_va)} imágenes)')
ax.hist(np.clip(d_tr, 0, tope), bins=bordes, weights=np.full(len(d_tr), 100 / len(d_tr)), histtype='step', color=TINTA, lw=1.0,
        label=f'Entrenamiento → entrenamiento ({len(d_tr)} imágenes)')
ax.set_xlabel('Diferencia absoluta media con la imagen más parecida (niveles de gris, miniaturas de 64 × 36)')
ax.set_ylabel('% de imágenes'); ax.yaxis.grid(True, color=REJILLA, lw=0.6); ax.set_axisbelow(True)
ax.legend(frameon=False, loc='upper right'); eje_con_coma(ax.xaxis, 0)
guardar(fig, DIR_FIG / 'Fig_5_7_pio_val_train.png'); plt.show()
VALORES['fig_5_7'] = detalle; print(detalle)

## 4. Guardar y descargar

In [ ]:
ruta_json = DIR_VAL / 'valores_cap05.json'
ruta_json.write_text(json.dumps(VALORES, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
ruta_zip = Path('/content/AgroVision_cap05.zip') if EN_COLAB else DIR_VAL / 'AgroVision_cap05.zip'
with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(DIR_FIG.glob('Fig_5_*.png')):
        z.write(f, 'cap05/' + f.name)
    for f in (ruta_json, DIR_VAL / 'pio_auditoria_por_imagen.csv', DIR_VAL / 'pio_vecino_val_train.csv'):
        z.write(f, f.name)
print('Figuras:', len(FIGURAS), '· zip de', round(ruta_zip.stat().st_size / 1e6, 1), 'MB')
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_zip))